<a href="https://colab.research.google.com/github/allaalmouiz/MedBot_LoRa/blob/main/MedBot_on_Custom_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#MedBot built on custom dataset
Submitted by: **`Alaa Almouiz F. Moh.`**

ID Number: **`S2026_176`**

Track: **Machine Learning**

For: **ZAKA AI, Inc. All Rights Reserved.©**

## **Problem Statement (Objective)**
The objective of this project is to create a simple QA LLM that can answer medical questionsand customize this LLM with any dataset.

**Just to give you a heads up:** We won't be having a model performing like ChatGPT or Bard, but at least we will have an idea about how we can create our own smaller versions of such powerful LLMs.  

## Importing and Installing Libraries/Packages
We will start by installing our necessary packages.

**bitsandbytes**: This package will allow us to run 4bit quantization on our model

**transformers**: This Hugging Face package will allow us to load state-of-the-art models easily into our notebook

**peft**: This package allows us to add PEFT techniques easily to our model, such as LoRA

**accelerate**: Accelerate is a handy package that allows us to run boiler plate code with a few lines of code

**datasets**: This package allows us to easily import datasets from the Hugging Face platform to be directly used

In [1]:
!pip install bitsandbytes
!pip install git+https://github.com/huggingface/transformers.git
!pip install git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/accelerate.git
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-0g9bgief
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-0g9bgief
  Resolved https://github.com/huggingface/transformers.git to commit 064f0e97c69ca2ac865be78ddff5ce73c54ab071
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.3.0.dev0-py3-none-any.whl size=11325556 sha256=bdcb65aa3fc932582da6547c27089f46662aa4e2eaa24fd31627a1b43593626d
  Stored in directory: /tmp/pip-ephem-wheel-cache-r13h4_k0/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
  

In [2]:
import torch
import transformers
from peft import prepare_model_for_kbit_training
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

## Loading our model

Let's start by loading our model. We will use the GPT Neox 20b Model by EleutherAI!

In [3]:
hf_model = "EleutherAI/gpt-neox-20b"

We will also set the bitsandbytes configurations needed for our model to run on our single colab GPU. The needed paramaters will be 'Double Quantization' 'Quantization Type' and the computational type needs to be set to bfloat16.

In [7]:
bitsbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                      bnb_4bit_use_double_quant=True,
                                      bnb_4bit_quant_type="nf4",
                                      bnb_4bit_compute_dtype=torch.float16)

🔮 **My Notes**

**Quantization** offers a solution to efficiently load and use large LLMs without compromising performance -> decrease high-precision weights and activations -> decrease Memory used.

**Configuration of the `BitsAndBytesConfig`**:  *Things I used*
* `load_in_4bit=True` - Enabling 4 bits Quantization (Trade-off between Size/Speed)
* `bnb_4bit_use_double_quant=True` - Applying second layer of quant to already quantized weights: "nested quantization"
* `bnb_4bit_compute_dtype=torch.float16` - Half precession for computation, high speed.
* `bnb_4bit_quant_type="nf4"` - Normal Float4

We will then set our tokenizer, and our model using the AutoTokenizer and AutoModelforCausalLM classes

In [13]:
#Test Your Zaka

# Tokenizer intialization
tokenizer = AutoTokenizer.from_pretrained(hf_model)

# Model intialization
model = AutoModelForCausalLM.from_pretrained(hf_model,
                                             device_map = 'auto',
                                             quantization_config= bitsbytes_config)

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

🔮 **My Notes**

* For the `tokenizer` I used the `AutoTokenizer` from pre-trained for my hugging face model, and applied End of sequence padding as well to it.
* For the model, I used the `AutoModelCausalLM` from pre-trained with my `bitsbytes_config` I early configured -> 4 bits.

## Model Preprocessing

We now have to apply some preprocessing to our model so we can prepare it for training. First we need to further reduce our memory consumption by using the gradient_checkpointing_enable() fucntion on our model. We then use the prepare_model_for_kbit_training function so that we can use 4bit quantization training.

In [ ]:
#Test Your Zaka

model.gradient_checkpointing_enable()           # Reduces memory by recomputing activations
model = prepare_model_for_kbit_training(model)  # Prepares model for 4-bit training

Explain with your own words how 4-bit quantization affects accuracy.

**Test your Zaka**

🔮 **My Notes**



We will also set a function that will print the number of trainable parameters our model has.

In [ ]:
def print_trainable_parameters(model):
    trainable_parameters = 0
    all_paramaters = 0
    for _, param in model.named_parameters():
        all_paramaters += param.numel()
        if param.requires_grad:
            trainable_parameters += param.numel()
    print(
        f"Trainable: {trainable_parameters} || All: {all_paramaters} || Trainable %: {100 * trainable_parameters / all_paramaters}"
    )

Finally we will set the configurations for our LoRA. The paramaters needed are the rank updates, the default LoRa alpha value, the target modules which need to be set to query_key_value, the default lora dropout rate, bias should be set to none, and the task type according to the model we are using.

In [ ]:
config = LoraConfig(
    #Test Your Zaka
    r = 2,
    lora_aplha,
    target_modules=['query_key_value'],
    lora_dropout=0.05, #reg
    bias= "none",
    task_type = TaskType.CAUSAL_LM
)

# Insert the configs above to the model using the get_peft_model function
#Test Your Zaka
model = get_peft_model(model, config)

# Print the trainable parameters of the model
print_trainable_parameters(model)

Trainable: 8650752 || All: 10597552128 || Trainable %: 0.08162971878329976


## Dataset Loading

Let's load our medical dataset from Hugging Face. We will use the `medalpaca/medical_meadow_wikidoc_patient_information` dataset. You can access it [here](https://huggingface.co/datasets/medalpaca/medical_meadow_wikidoc).

In [ ]:
#Test Your Zaka

# Loading the datset
data = load_dataset("medalpaca/medical_meadow_wikidoc_patient_information")

# Mapping the needed column as our data using a lambda statement
data = data.map(lambda samples: tokenizer(samples["output"]), batched=True)

## Model Training and Testing

Now we train the model usig the transformers library. Before doing so, we set the tokenizer to be the end of sequence tokens since it is required by our model. Your goal here is to tune the paramaters until you get a running model on a single colab GPU.

In [ ]:
    args=transformers.TrainingArguments(
        gradient_accumulation_steps=4,      # Accumulate gradients over 4 steps (simulates larger batch)
        warmup_steps=2,                     # Gradual LR warmup for training stability
        max_steps=10,                       # Total training steps (keep low for testing on Colab)
    )

In [ ]:
# Setting the tokenizer padding to be 'eos' tokens
tokenizer.pad_token = tokenizer.eos_token

training_args = TrainingArguments(
    output_dir = "./results",
    per_device_train_batch_size = 2,
    num_train_epochs=3,
    logging_steps = 5,
    save_steps = 10,
    save_total_limit = 1, # keep the latest checkpoint
    learning_rate=2e-4,
    fp16 = True #enables 16bit trainig for efficency
)

trainer = transformers.Trainer(
    model = model,
    args = training_args,
    train_dataset = data["train"],
    processing_class = tokenizer,
    data_collator = data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# This silences the warnings
model.config.use_cache = False

# Train the model!
#Test Your Zaka
trainer.train()

Explain 4 of the training arguments you used in your Trainer, how they are used, and what do they represent

**Test your Zaka**

🔮 **My Notes:**

We now save our model as a pretrained version so that we can set the LoRA configurations. This model will be saved to a separate folder on the next block.

In [ ]:
#Test Your Zaka

trainer.save_model("outputs")

saved_model = model.merge_and_unload() if hasattr(model, "merge_and_unload") else model
saved_model.save_pretrained("outputs")

Before testing our model, we have to get the LoRA configs from our pre-trained model and set them to our new model using the get_peft_model() function.

In [ ]:
#Test Your Zaka

lora_configs = LoraConfig.from_pretrained("outputs")
model = get_peft_model(model, lora_configs)

We need to set our prompt as a variable, and also our device currently in use.

In [ ]:
#Test Your Zaka

prompt = "____"
device = "cuda:0"

Finally, we will make our LLM generate text based on the data. First we user the tokenizer() function on our prompt.

In [ ]:
#Test Your Zaka
inputs = tokenizer(prompt, return_tensors="pt").to(device)

Let's now use the generate() function on our model, and print the decoded version of our output.

In [ ]:
outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))